## 0. 환경 설정

In [41]:
!pip install lightgbm scikit-learn joblib --quiet


In [42]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, fbeta_score,
    mean_absolute_error, mean_squared_error,
)
from sklearn.isotonic import IsotonicRegression

pd.set_option('display.max_columns', 100)
print('lightgbm version:', lgb.__version__)


lightgbm version: 4.7.0


In [43]:
from pathlib import Path

CUR_DIR = Path.cwd().resolve()
BASE_DIR = CUR_DIR / "data"

In [44]:
DATA_PATH = BASE_DIR / "1단계_학습데이터_최종.csv"

CHECKPOINT_ROOT = CUR_DIR / "stage1_hurdle_v6"

RESULT_DIR = CUR_DIR / "result" / "step1_model"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("입력 데이터:", DATA_PATH)
print("체크포인트:", CHECKPOINT_ROOT)
print("결과 저장 폴더:", RESULT_DIR)

입력 데이터: .\data\1단계_학습데이터_최종.csv
체크포인트: .\stage1_hurdle_v6
결과 저장 폴더: .\result\step1_model


## 1. 데이터 로드 및 피처 정의

In [45]:
df = pd.read_csv(DATA_PATH)
print('전체 shape:', df.shape)
print('분기 범위:', df['기준_년분기_코드'].min(), '~', df['기준_년분기_코드'].max())
df.head(3)


전체 shape: (1441629, 74)
분기 범위: 20211 ~ 20253


,기준_년분기_코드,상권_구분_코드,상권_코드,서비스_업종_코드,점포_수,유사_업종_점포_수,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수,당월_매출_금액,당월_매출_건수,지출_총금액,총_유동인구_수,연령대_10_유동인구_수,연령대_20_유동인구_수,연령대_30_유동인구_수,연령대_40_유동인구_수,연령대_50_유동인구_수,연령대_60_이상_유동인구_수,시간대_00_06_유동인구_수,시간대_06_11_유동인구_수,시간대_11_14_유동인구_수,시간대_14_17_유동인구_수,시간대_17_21_유동인구_수,시간대_21_24_유동인구_수,총_상주인구_수,총_가구_수,총_직장_인구_수,남성_직장_인구_수,여성_직장_인구_수,연령대_10_직장_인구_수,연령대_20_직장_인구_수,연령대_30_직장_인구_수,연령대_40_직장_인구_수,연령대_50_직장_인구_수,연령대_60_이상_직장_인구_수,집객시설_수,매출결측유형,집객시설정보없음,유동인구정보없음,직장인구정보없음,상주인구정보없음,소비정보없음,y_다음분기_폐업률,개업률,순증감_점포수,프랜차이즈_비율,객단가,주말_매출_비율,점포당_매출,매출_성비,식료품_지출_비율,의류_신발_지출_비율,생활용품_지출_비율,의료비_지출_비율,교통_지출_비율,여가_지출_비율,문화_지출_비율,교육_지출_비율,유흥_지출_비율,지출_대비_매출_비율,심야_유동인구_비율,20대_유동인구_비율,60대이상_유동인구_비율,유동인구_대비_점포수,직장_상주_비율,여성_상주인구_비율,10대_상주인구_비율,20대_상주인구_비율,30대_상주인구_비율,40대_상주인구_비율,50대_상주인구_비율,60_이상대_상주인구_비율
0,20211,A,3110001,CS100001,12,12,0,0,0,0,382839289.0,13701.0,520427635.0,166405.0,30014.0,17921.0,16183.0,26627.0,29059.0,46601.0,41569.0,35726.0,19802.0,20188.0,27709.0,21413.0,1020.0,401.0,2419.0,1357.0,1062.0,2.0,299.0,648.0,793.0,526.0,151.0,12.0,0,0,0,0,0,0,0.0,0.0,0,0.00,27942.434056,0.385359,3.190327e+07,0.681490,0.248115,0.124616,0.071331,0.12677,0.169752,0.042703,0.028163,0.14639,0.042161,0.735624,0.12868,0.107695,0.280046,0.000072,2.371569,0.543137,0.156863,0.139216,0.092157,0.156863,0.190196,0.264706
1,20211,A,3110001,CS100008,3,3,0,0,0,0,122957138.0,12039.0,520427635.0,166405.0,30014.0,17921.0,16183.0,26627.0,29059.0,46601.0,41569.0,35726.0,19802.0,20188.0,27709.0,21413.0,1020.0,401.0,2419.0,1357.0,1062.0,2.0,299.0,648.0,793.0,526.0,151.0,12.0,0,0,0,0,0,0,0.0,0.0,0,0.00,10213.235152,0.503322,4.098571e+07,0.573113,0.248115,0.124616,0.071331,0.12677,0.169752,0.042703,0.028163,0.14639,0.042161,0.236262,0.12868,0.107695,0.280046,0.000018,2.371569,0.543137,0.156863,0.139216,0.092157,0.156863,0.190196,0.264706
2,20211,A,3110001,CS100009,3,4,0,0,0,1,19012827.0,264.0,520427635.0,166405.0,30014.0,17921.0,16183.0,26627.0,29059.0,46601.0,41569.0,35726.0,19802.0,20188.0,27709.0,21413.0,1020.0,401.0,2419.0,1357.0,1062.0,2.0,299.0,648.0,793.0,526.0,151.0,12.0,0,0,0,0,0,0,0.0,0.0,0,0.25,72018.284091,0.246670,4.753207e+06,0.774278,0.248115,0.124616,0.071331,0.12677,0.169752,0.042703,0.028163,0.14639,0.042161,0.036533,0.12868,0.107695,0.280046,0.000024,2.371569,0.543137,0.156863,0.139216,0.092157,0.156863,0.190196,0.264706


In [46]:
TARGET_COL = 'y_다음분기_폐업률'
ID_COLS = ['기준_년분기_코드', '상권_코드']
CATEGORICAL_COLS = ['상권_구분_코드', '서비스_업종_코드']

for col in CATEGORICAL_COLS:
    df[col] = df[col].astype('category')

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]
print(f'피처 개수: {len(FEATURE_COLS)}')
print(f'범주형 피처: {CATEGORICAL_COLS}')


피처 개수: 71
범주형 피처: ['상권_구분_코드', '서비스_업종_코드']


### 1-1. 모멘텀·변화율 피처

In [47]:
df = df.sort_values(['상권_코드', '서비스_업종_코드', '기준_년분기_코드']).reset_index(drop=True)
_group = df.groupby(['상권_코드', '서비스_업종_코드'])

MOMENTUM_SRC_COLS = [
    '당월_매출_금액', '총_유동인구_수', '점포_수', '폐업_률',
    '유사_업종_점포_수', '개업_점포_수',
]
MOMENTUM_SRC_COLS = [c for c in MOMENTUM_SRC_COLS if c in df.columns]
print('모멘텀 피처를 생성할 원본 컬럼:', MOMENTUM_SRC_COLS)

for col in MOMENTUM_SRC_COLS:
    df[f'{col}_qoq_pct'] = _group[col].pct_change().replace([np.inf, -np.inf], np.nan)
    df[f'{col}_lag1'] = _group[col].shift(1)
    df[f'{col}_4q_mean'] = _group[col].transform(lambda s: s.rolling(4, min_periods=2).mean())
    df[f'{col}_vs_4q_mean'] = df[col] / df[f'{col}_4q_mean'].replace(0, np.nan) - 1

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]
new_momentum_cols = [c for c in FEATURE_COLS if any(c.startswith(f'{s}_') for s in MOMENTUM_SRC_COLS)]
print(f'모멘텀 피처 추가 후 전체 피처 개수: {len(FEATURE_COLS)} (신규 {len(new_momentum_cols)}개)')
print('신규 피처 예시:', new_momentum_cols[:8])


모멘텀 피처를 생성할 원본 컬럼: ['당월_매출_금액', '총_유동인구_수', '점포_수', '폐업_률', '유사_업종_점포_수', '개업_점포_수']


모멘텀 피처 추가 후 전체 피처 개수: 95 (신규 24개)
신규 피처 예시: ['당월_매출_금액_qoq_pct', '당월_매출_금액_lag1', '당월_매출_금액_4q_mean', '당월_매출_금액_vs_4q_mean', '총_유동인구_수_qoq_pct', '총_유동인구_수_lag1', '총_유동인구_수_4q_mean', '총_유동인구_수_vs_4q_mean']


### 1-2. 계절성 피처 + Shrinkage 과거 이력 피처 + 상호작용 피처

In [48]:
df['분기_Q'] = (df['기준_년분기_코드'] % 10).astype('category')
df['연도'] = (df['기준_년분기_코드'] // 10).astype(int)
CATEGORICAL_COLS = CATEGORICAL_COLS + ['분기_Q']

quarterly_mean = df.groupby('기준_년분기_코드')[TARGET_COL].mean().sort_index()
quarterly_global_prior = quarterly_mean.expanding().mean().shift(1)  
df['글로벌_과거평균폐업률'] = df['기준_년분기_코드'].map(quarterly_global_prior).fillna(0.0)

SHRINKAGE_K = 20  

def shrinkage_expanding_mean(group_target, prior):
    cum_sum = group_target.shift(1).expanding().sum()
    cum_count = group_target.shift(1).expanding().count()
    return (cum_sum.fillna(0.0) + SHRINKAGE_K * prior) / (cum_count.fillna(0.0) + SHRINKAGE_K)

_grp_area_service = df.groupby(['상권_코드', '서비스_업종_코드'])[TARGET_COL]
_cum_sum_as = _grp_area_service.transform(lambda s: s.shift(1).expanding().sum())
_cum_count_as = _grp_area_service.transform(lambda s: s.shift(1).expanding().count())
df['상권업종_과거평균폐업률'] = (
    (_cum_sum_as.fillna(0.0) + SHRINKAGE_K * df['글로벌_과거평균폐업률'])
    / (_cum_count_as.fillna(0.0) + SHRINKAGE_K)
)

_grp_area = df.groupby('상권_코드')[TARGET_COL]
_cum_sum_a = _grp_area.transform(lambda s: s.shift(1).expanding().sum())
_cum_count_a = _grp_area.transform(lambda s: s.shift(1).expanding().count())
df['상권_과거평균폐업률'] = (
    (_cum_sum_a.fillna(0.0) + SHRINKAGE_K * df['글로벌_과거평균폐업률'])
    / (_cum_count_a.fillna(0.0) + SHRINKAGE_K)
)

if '프랜차이즈_비율' in df.columns and '유사_업종_점포_수' in df.columns:
    df['프랜차이즈비율_x_경쟁강도'] = df['프랜차이즈_비율'] * df['유사_업종_점포_수']

if '상권_과거평균폐업률' in df.columns and '당월_매출_금액_qoq_pct' in df.columns:
    df['과거폐업률_x_매출모멘텀'] = df['상권_과거평균폐업률'] * df['당월_매출_금액_qoq_pct']

if '상권업종_과거평균폐업률' in df.columns and '총_유동인구_수_qoq_pct' in df.columns:
    df['과거폐업률_x_유동인구모멘텀'] = df['상권업종_과거평균폐업률'] * df['총_유동인구_수_qoq_pct']

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]
print(f'계절성/이력(shrinkage)/상호작용 피처 추가 후 전체 피처 개수: {len(FEATURE_COLS)}')
print('범주형 피처:', CATEGORICAL_COLS)
print()
print('상권업종_과거평균폐업률 결측 비율(이제 shrinkage로 0건이어야 정상):',
      df['상권업종_과거평균폐업률'].isna().mean().round(4))


계절성/이력(shrinkage)/상호작용 피처 추가 후 전체 피처 개수: 103
범주형 피처: ['상권_구분_코드', '서비스_업종_코드', '분기_Q']

상권업종_과거평균폐업률 결측 비율(이제 shrinkage로 0건이어야 정상): 0.0


## 2. 분기 코드 헬퍼 & 폴드 목록

In [49]:
def q_to_idx(q: int) -> int:
    year, quarter = divmod(q, 10)
    return year * 4 + (quarter - 1)

def idx_to_q(idx: int) -> int:
    year, quarter = divmod(idx, 4)
    return year * 10 + (quarter + 1)

def prev_q(q: int) -> int:
    return idx_to_q(q_to_idx(q) - 1)

def next_q(q: int) -> int:
    return idx_to_q(q_to_idx(q) + 1)

assert prev_q(20221) == 20214
assert next_q(20214) == 20221
print('분기 헬퍼 함수 정상 동작')


분기 헬퍼 함수 정상 동작


In [50]:
FIRST_QUARTER = 20211         
FIRST_TARGET_QUARTER = 20221   
LAST_TARGET_QUARTER = 20244    

TARGET_QUARTERS = [
    idx_to_q(i) for i in range(q_to_idx(FIRST_TARGET_QUARTER), q_to_idx(LAST_TARGET_QUARTER) + 1)
]

print(f'총 폴드 수: {len(TARGET_QUARTERS)}')
print('예측 대상 분기 목록:', TARGET_QUARTERS)


총 폴드 수: 12
예측 대상 분기 목록: [20221, 20222, 20223, 20224, 20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244]


In [51]:
N_INNER_VAL_QUARTERS = 2

def build_fold(test_target_quarter):
    test_snapshot_quarter = prev_q(test_target_quarter)     
    train_end_quarter = prev_q(test_snapshot_quarter)       

    train_idx_range = range(q_to_idx(FIRST_QUARTER), q_to_idx(train_end_quarter) + 1)
    train_quarters = [idx_to_q(i) for i in train_idx_range]

    n_val = min(N_INNER_VAL_QUARTERS, len(train_quarters) - 1)
    n_val = max(n_val, 1)
    inner_val_quarters = train_quarters[-n_val:]
    inner_train_quarters = train_quarters[:-n_val]

    return {
        'test_target_quarter': test_target_quarter,
        'test_snapshot_quarter': test_snapshot_quarter,
        'train_quarters': train_quarters,
        'inner_train_quarters': inner_train_quarters,
        'inner_val_quarters': inner_val_quarters,
    }

print('첫 폴드 :', build_fold(TARGET_QUARTERS[0]))
print('마지막 폴드 :', build_fold(TARGET_QUARTERS[-1]))


첫 폴드 : {'test_target_quarter': 20221, 'test_snapshot_quarter': 20214, 'train_quarters': [20211, 20212, 20213], 'inner_train_quarters': [20211], 'inner_val_quarters': [20212, 20213]}
마지막 폴드 : {'test_target_quarter': 20244, 'test_snapshot_quarter': 20243, 'train_quarters': [20211, 20212, 20213, 20214, 20221, 20222, 20223, 20224, 20231, 20232, 20233, 20234, 20241, 20242], 'inner_train_quarters': [20211, 20212, 20213, 20214, 20221, 20222, 20223, 20224, 20231, 20232, 20233, 20234], 'inner_val_quarters': [20241, 20242]}


## 3. 데이터 분할 & 학습 함수

In [52]:
def make_xy(quarters):
    sub = df[df['기준_년분기_코드'].isin(quarters)]
    return sub[FEATURE_COLS], sub[TARGET_COL]

def to_binary(y):
    return (y > 0).astype(int)

def positive_subset(X, y):
    mask = y > 0
    return X[mask], y[mask]

def find_best_threshold(y_true, y_prob, beta=2.0):
    thresholds = np.linspace(0.05, 0.95, 19)
    scores = [
        fbeta_score(y_true, (y_prob >= t).astype(int), beta=beta, zero_division=0)
        for t in thresholds
    ]
    return float(thresholds[int(np.argmax(scores))])

SEEDS = [42, 202, 777, 1234, 31415]


In [53]:
LGB_COMMON_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
)

SCALE_POS_WEIGHT_CANDIDATES = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

def train_or_load_stage_a(fold, force_retrain=False):
    q = fold['test_target_quarter']
    stage_dir = f"{CHECKPOINT_ROOT}/target_{q}/stage_a_classifier"
    os.makedirs(stage_dir, exist_ok=True)
    bundle_path = f'{stage_dir}/bundle.pkl'   
    meta_path = f'{stage_dir}/meta.json'

    if (not force_retrain) and os.path.exists(bundle_path):
        bundle = joblib.load(bundle_path)
        with open(meta_path, encoding='utf-8') as f:
            meta = json.load(f)
        print(f'  [Stage A] target={q} 체크포인트 로드 '
              f'(scale_pos_weight={meta["best_scale_pos_weight"]}, '
              f'threshold={meta["best_threshold"]:.2f}, seeds={len(bundle["models"])}개)')
        return bundle, meta

    X_inner_train, y_inner_train = make_xy(fold['inner_train_quarters'])
    X_inner_val, y_inner_val = make_xy(fold['inner_val_quarters'])
    X_train_full, y_train_full = make_xy(fold['train_quarters'])

    yb_inner_train = to_binary(y_inner_train)
    yb_inner_val = to_binary(y_inner_val)
    yb_train_full = to_binary(y_train_full)

    best_w, best_score, best_iter_ref = None, -np.inf, None
    for w in SCALE_POS_WEIGHT_CANDIDATES:
        m = lgb.LGBMClassifier(objective='binary', scale_pos_weight=w, random_state=42, **LGB_COMMON_PARAMS)
        m.fit(
            X_inner_train, yb_inner_train,
            eval_set=[(X_inner_val, yb_inner_val)],
            eval_metric='auc',
            categorical_feature=CATEGORICAL_COLS,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
        p_val = m.predict_proba(X_inner_val)[:, 1]
        score = average_precision_score(yb_inner_val, p_val)
        if score > best_score:
            best_w, best_score, best_iter_ref = w, score, m.best_iteration_

    inner_models, inner_best_iters = [], []
    for seed in SEEDS:
        m = lgb.LGBMClassifier(objective='binary', scale_pos_weight=best_w, random_state=seed, **LGB_COMMON_PARAMS)
        m.fit(
            X_inner_train, yb_inner_train,
            eval_set=[(X_inner_val, yb_inner_val)],
            eval_metric='auc',
            categorical_feature=CATEGORICAL_COLS,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
        inner_models.append(m)
        inner_best_iters.append(m.best_iteration_)

    p_val_ensemble = np.mean([m.predict_proba(X_inner_val)[:, 1] for m in inner_models], axis=0)

    isotonic = IsotonicRegression(out_of_bounds='clip')
    isotonic.fit(p_val_ensemble, yb_inner_val)
    p_val_calibrated = isotonic.predict(p_val_ensemble)

    best_threshold = find_best_threshold(yb_inner_val, p_val_calibrated, beta=2.0)

    final_models = []
    for seed, best_iter in zip(SEEDS, inner_best_iters):
        final_params = dict(LGB_COMMON_PARAMS)
        final_params['n_estimators'] = max(best_iter, 1)
        m = lgb.LGBMClassifier(objective='binary', scale_pos_weight=best_w, random_state=seed, **final_params)
        m.fit(X_train_full, yb_train_full, categorical_feature=CATEGORICAL_COLS)
        final_models.append(m)

    bundle = {'models': final_models, 'isotonic': isotonic}
    joblib.dump(bundle, bundle_path)
    meta = {
        'best_iterations': [int(x) for x in inner_best_iters],
        'best_scale_pos_weight': best_w,
        'best_threshold': best_threshold,
        'seeds': SEEDS,
        **fold,
    }
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f'  [Stage A] target={q} 학습 완료 '
          f'(scale_pos_weight={best_w}, threshold={best_threshold:.2f}, seeds={len(final_models)}개)')
    return bundle, meta


In [54]:
STAGE_B_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
)

REG_OBJECTIVE = 'huber'
HUBER_ALPHA = 0.9  

CLIP_PERCENTILE_CANDIDATES = [0.97, 0.99, 0.995, 1.0]

TARGET_WINSORIZE_PERCENTILE = 0.99

def train_or_load_stage_b(fold, force_retrain=False):
    q = fold['test_target_quarter']
    stage_dir = f"{CHECKPOINT_ROOT}/target_{q}/stage_b_regressor"
    os.makedirs(stage_dir, exist_ok=True)
    bundle_path = f'{stage_dir}/bundle.pkl'   
    meta_path = f'{stage_dir}/meta.json'

    if (not force_retrain) and os.path.exists(bundle_path):
        bundle = joblib.load(bundle_path)
        with open(meta_path, encoding='utf-8') as f:
            meta = json.load(f)
        print(f'  [Stage B] target={q} 체크포인트 로드 '
              f'(clip_percentile={meta.get("clip_percentile")}, '
              f'clip_upper_log={meta["clip_upper_log"]:.2f}, seeds={len(bundle["models"])}개)')
        return bundle, meta

    X_inner_train, y_inner_train = make_xy(fold['inner_train_quarters'])
    X_inner_val, y_inner_val = make_xy(fold['inner_val_quarters'])
    X_train_full, y_train_full = make_xy(fold['train_quarters'])

    X_inner_train_pos, y_inner_train_pos = positive_subset(X_inner_train, y_inner_train)
    X_inner_val_pos, y_inner_val_pos = positive_subset(X_inner_val, y_inner_val)
    X_train_full_pos, y_train_full_pos = positive_subset(X_train_full, y_train_full)

    y_inner_train_pos_log = np.log1p(y_inner_train_pos)
    y_inner_val_pos_log = np.log1p(y_inner_val_pos)
    y_train_full_pos_log = np.log1p(y_train_full_pos)

    inner_train_winsor_upper = np.quantile(y_inner_train_pos_log, TARGET_WINSORIZE_PERCENTILE)
    y_inner_train_pos_log_w = np.clip(y_inner_train_pos_log, None, inner_train_winsor_upper)

    train_full_winsor_upper = np.quantile(y_train_full_pos_log, TARGET_WINSORIZE_PERCENTILE)
    y_train_full_pos_log_w = np.clip(y_train_full_pos_log, None, train_full_winsor_upper)

    best_clip_pct, best_val_mae, best_iter_ref = None, np.inf, None
    for pct in CLIP_PERCENTILE_CANDIDATES:
        m = lgb.LGBMRegressor(objective=REG_OBJECTIVE, alpha=HUBER_ALPHA, random_state=42, **STAGE_B_PARAMS)
        m.fit(
            X_inner_train_pos, y_inner_train_pos_log_w,
            eval_set=[(X_inner_val_pos, y_inner_val_pos_log)],
            eval_metric='mae',
            categorical_feature=CATEGORICAL_COLS,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
        pred_val_log_raw = m.predict(X_inner_val_pos)

        if pct < 1.0:
            clip_val = float(np.quantile(y_inner_train_pos_log_w, pct))
            pred_val_log_clipped = np.clip(pred_val_log_raw, None, clip_val)
        else:
            pred_val_log_clipped = pred_val_log_raw

        pred_val_original = np.expm1(pred_val_log_clipped)
        val_mae = mean_absolute_error(y_inner_val_pos, pred_val_original)

        if val_mae < best_val_mae:
            best_clip_pct, best_val_mae, best_iter_ref = pct, val_mae, m.best_iteration_

    inner_models, inner_best_iters = [], []
    for seed in SEEDS:
        m = lgb.LGBMRegressor(objective=REG_OBJECTIVE, alpha=HUBER_ALPHA, random_state=seed, **STAGE_B_PARAMS)
        m.fit(
            X_inner_train_pos, y_inner_train_pos_log_w,
            eval_set=[(X_inner_val_pos, y_inner_val_pos_log)],
            eval_metric='mae',
            categorical_feature=CATEGORICAL_COLS,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
        inner_models.append(m)
        inner_best_iters.append(m.best_iteration_)

    final_models = []
    for seed, best_iter in zip(SEEDS, inner_best_iters):
        final_params = dict(STAGE_B_PARAMS)
        final_params['n_estimators'] = max(best_iter, 1)
        m = lgb.LGBMRegressor(objective=REG_OBJECTIVE, alpha=HUBER_ALPHA, random_state=seed, **final_params)
        m.fit(X_train_full_pos, y_train_full_pos_log_w, categorical_feature=CATEGORICAL_COLS)
        final_models.append(m)

    if best_clip_pct < 1.0:
        clip_upper_log = float(np.quantile(y_train_full_pos_log_w, best_clip_pct))
    else:
        clip_upper_log = float(y_train_full_pos_log_w.max())  

    bundle = {'models': final_models}
    joblib.dump(bundle, bundle_path)
    meta = {
        'best_iterations': [int(x) for x in inner_best_iters],
        'objective': REG_OBJECTIVE,
        'huber_alpha': HUBER_ALPHA,
        'clip_percentile': best_clip_pct,
        'clip_upper_log': clip_upper_log,
        'target_winsorize_percentile': TARGET_WINSORIZE_PERCENTILE,
        'train_full_winsor_upper_log': float(train_full_winsor_upper),
        'inner_val_mae_positive_subset': float(best_val_mae),
        'seeds': SEEDS,
        **fold,
    }
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f'  [Stage B] target={q} 학습 완료 '
          f'(objective={REG_OBJECTIVE}, clip_percentile={best_clip_pct}, '
          f'clip_upper_log={clip_upper_log:.2f} -> 원래 스케일 상한 {np.expm1(clip_upper_log):.2f}, '
          f'inner_val_mae_positive_subset={best_val_mae:.4f}, seeds={len(final_models)}개)')
    return bundle, meta


In [55]:
def evaluate_fold(fold, clf_bundle, clf_meta, reg_bundle, reg_meta):
    q_target = fold['test_target_quarter']
    q_snapshot = fold['test_snapshot_quarter']

    X_test, y_test = make_xy([q_snapshot])
    id_cols_test = df.loc[X_test.index, ['상권_코드', '서비스_업종_코드']].copy()

    p_nonzero_raw = np.mean([m.predict_proba(X_test)[:, 1] for m in clf_bundle['models']], axis=0)
    p_nonzero = clf_bundle['isotonic'].predict(p_nonzero_raw)

    expected_positive_raw_log = np.mean([m.predict(X_test) for m in reg_bundle['models']], axis=0)
    clip_upper_log = reg_meta['clip_upper_log']
    expected_positive_log_clipped = np.clip(expected_positive_raw_log, None, clip_upper_log)

    expected_positive_raw = np.expm1(expected_positive_raw_log)
    expected_positive = np.expm1(expected_positive_log_clipped)

    y_pred_hurdle = p_nonzero * expected_positive

    y_test_bin = to_binary(y_test)

    threshold = clf_meta.get('best_threshold', 0.5)
    pred_label = (p_nonzero >= threshold).astype(int)

    roc_auc = roc_auc_score(y_test_bin, p_nonzero)
    pr_auc = average_precision_score(y_test_bin, p_nonzero)
    precision = precision_score(y_test_bin, pred_label, zero_division=0)
    recall = recall_score(y_test_bin, pred_label, zero_division=0)
    f1 = f1_score(y_test_bin, pred_label, zero_division=0)

    mae_overall = mean_absolute_error(y_test, y_pred_hurdle)
    rmse_overall = np.sqrt(mean_squared_error(y_test, y_pred_hurdle))

    baseline_zero = np.zeros_like(y_test.values, dtype=float)
    mae_baseline = mean_absolute_error(y_test, baseline_zero)

    mask_pos = y_test.values > 0
    if mask_pos.sum() > 0:
        mae_pos = mean_absolute_error(y_test.values[mask_pos], y_pred_hurdle[mask_pos])
        rmse_pos = np.sqrt(mean_squared_error(y_test.values[mask_pos], y_pred_hurdle[mask_pos]))
        mae_pos_baseline = mean_absolute_error(y_test.values[mask_pos], baseline_zero[mask_pos])
    else:
        mae_pos = rmse_pos = mae_pos_baseline = np.nan

    fold_result = {
        'test_target_quarter': q_target,
        'test_snapshot_quarter': q_snapshot,
        'n_train': sum(len(make_xy([qq])[1]) for qq in fold['train_quarters']),
        'n_test': len(y_test),
        'scale_pos_weight': clf_meta.get('best_scale_pos_weight'),
        'threshold_used': threshold,
        'reg_objective': reg_meta.get('objective'),
        'clip_percentile': reg_meta.get('clip_percentile'),
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'mae_overall': mae_overall,
        'rmse_overall': rmse_overall,
        'mae_zero_baseline': mae_baseline,
        'mae_positive_subset': mae_pos,
        'rmse_positive_subset': rmse_pos,
        'mae_positive_subset_zero_baseline': mae_pos_baseline,
        'n_positive_subset': int(mask_pos.sum()),
        'clip_upper_log': clip_upper_log,
        'clip_upper_original_scale': float(np.expm1(clip_upper_log)),
        'clipped_ratio': float((expected_positive_raw_log > clip_upper_log).mean()),
    }

    pred_table = id_cols_test.copy()
    pred_table['예측_대상_분기'] = q_target
    pred_table['실제_폐업률'] = y_test.values
    pred_table['예측_폐업위험점수'] = y_pred_hurdle
    pred_table['p_nonzero'] = p_nonzero
    pred_table['p_nonzero_raw'] = p_nonzero_raw
    pred_table['threshold_used'] = threshold
    pred_table['expected_if_positive'] = expected_positive
    pred_table['expected_if_positive_raw'] = expected_positive_raw

    return fold_result, pred_table


## 4. 메인 루프 — 전체 폴드 순차 실행

In [56]:
FORCE_RETRAIN = False  

fold_results = []
oof_prediction_tables = []

for q_target in TARGET_QUARTERS:
    print(f'\n===== 폴드: 예측 대상 {q_target} =====')
    fold = build_fold(q_target)
    print(f'  학습 분기: {fold["train_quarters"][0]} ~ {fold["train_quarters"][-1]} '
          f'(내부검증={fold["inner_val_quarters"]}) / 테스트 스냅샷={fold["test_snapshot_quarter"]}')

    clf_bundle, clf_meta = train_or_load_stage_a(fold, force_retrain=FORCE_RETRAIN)
    reg_bundle, reg_meta = train_or_load_stage_b(fold, force_retrain=FORCE_RETRAIN)

    fold_result, pred_table = evaluate_fold(fold, clf_bundle, clf_meta, reg_bundle, reg_meta)
    fold_results.append(fold_result)
    oof_prediction_tables.append(pred_table)

    print(f'  ROC-AUC={fold_result["roc_auc"]:.4f}  PR-AUC={fold_result["pr_auc"]:.4f}  '
          f'Recall={fold_result["recall"]:.4f}  scale_pos_weight={fold_result["scale_pos_weight"]}  '
          f'threshold={fold_result["threshold_used"]:.2f}  '
          f'MAE(양성subset)={fold_result["mae_positive_subset"]:.4f}')

print('\n전체 폴드 실행 완료.')



===== 폴드: 예측 대상 20221 =====
  학습 분기: 20211 ~ 20213 (내부검증=[20212, 20213]) / 테스트 스냅샷=20214
  [Stage A] target=20221 체크포인트 로드 (scale_pos_weight=1.5, threshold=0.10, seeds=5개)
  [Stage B] target=20221 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


  ROC-AUC=0.8539  PR-AUC=0.4318  Recall=0.8988  scale_pos_weight=1.5  threshold=0.10  MAE(양성subset)=17.9830

===== 폴드: 예측 대상 20222 =====
  학습 분기: 20211 ~ 20214 (내부검증=[20213, 20214]) / 테스트 스냅샷=20221
  [Stage A] target=20222 체크포인트 로드 (scale_pos_weight=1.0, threshold=0.10, seeds=5개)
  [Stage B] target=20222 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


  ROC-AUC=0.8648  PR-AUC=0.4379  Recall=0.8744  scale_pos_weight=1.0  threshold=0.10  MAE(양성subset)=17.2114

===== 폴드: 예측 대상 20223 =====
  학습 분기: 20211 ~ 20221 (내부검증=[20214, 20221]) / 테스트 스냅샷=20222
  [Stage A] target=20223 체크포인트 로드 (scale_pos_weight=1.0, threshold=0.10, seeds=5개)
  [Stage B] target=20223 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


  ROC-AUC=0.8656  PR-AUC=0.5139  Recall=0.8189  scale_pos_weight=1.0  threshold=0.10  MAE(양성subset)=20.0076

===== 폴드: 예측 대상 20224 =====
  학습 분기: 20211 ~ 20222 (내부검증=[20221, 20222]) / 테스트 스냅샷=20223
  [Stage A] target=20224 체크포인트 로드 (scale_pos_weight=1.5, threshold=0.10, seeds=5개)
  [Stage B] target=20224 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


  ROC-AUC=0.8643  PR-AUC=0.4882  Recall=0.8650  scale_pos_weight=1.5  threshold=0.10  MAE(양성subset)=17.5141

===== 폴드: 예측 대상 20231 =====
  학습 분기: 20211 ~ 20223 (내부검증=[20222, 20223]) / 테스트 스냅샷=20224
  [Stage A] target=20231 체크포인트 로드 (scale_pos_weight=1.5, threshold=0.10, seeds=5개)
  [Stage B] target=20231 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


  ROC-AUC=0.8605  PR-AUC=0.5126  Recall=0.8765  scale_pos_weight=1.5  threshold=0.10  MAE(양성subset)=18.4545

===== 폴드: 예측 대상 20232 =====
  학습 분기: 20211 ~ 20224 (내부검증=[20223, 20224]) / 테스트 스냅샷=20231
  [Stage A] target=20232 체크포인트 로드 (scale_pos_weight=1.5, threshold=0.10, seeds=5개)
  [Stage B] target=20232 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


... (3 outputs left out)

  ROC-AUC=0.8623  PR-AUC=0.5195  Recall=0.8462  scale_pos_weight=1.0  threshold=0.10  MAE(양성subset)=19.2444

===== 폴드: 예측 대상 20242 =====
  학습 분기: 20211 ~ 20234 (내부검증=[20233, 20234]) / 테스트 스냅샷=20241
  [Stage A] target=20242 체크포인트 로드 (scale_pos_weight=1.0, threshold=0.10, seeds=5개)
  [Stage B] target=20242 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


  ROC-AUC=0.8643  PR-AUC=0.5016  Recall=0.9041  scale_pos_weight=1.0  threshold=0.10  MAE(양성subset)=17.8161

===== 폴드: 예측 대상 20243 =====
  학습 분기: 20211 ~ 20241 (내부검증=[20234, 20241]) / 테스트 스냅샷=20242
  [Stage A] target=20243 체크포인트 로드 (scale_pos_weight=1.5, threshold=0.15, seeds=5개)
  [Stage B] target=20243 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


  ROC-AUC=0.8657  PR-AUC=0.5051  Recall=0.8521  scale_pos_weight=1.5  threshold=0.15  MAE(양성subset)=18.1870

===== 폴드: 예측 대상 20244 =====
  학습 분기: 20211 ~ 20242 (내부검증=[20241, 20242]) / 테스트 스냅샷=20243
  [Stage A] target=20244 체크포인트 로드 (scale_pos_weight=1.0, threshold=0.15, seeds=5개)
  [Stage B] target=20244 체크포인트 로드 (clip_percentile=0.97, clip_upper_log=4.62, seeds=5개)


  ROC-AUC=0.8748  PR-AUC=0.5095  Recall=0.8108  scale_pos_weight=1.0  threshold=0.15  MAE(양성subset)=17.8117

전체 폴드 실행 완료.


## 5. 폴드별 성능 요약

In [57]:
fold_summary_df = pd.DataFrame(fold_results)
fold_summary_path = f'{RESULT_DIR}/fold_performance_summary.csv'
fold_summary_df.to_csv(fold_summary_path, index=False, encoding='utf-8-sig')

print(f'폴드별 성능 요약 저장: {fold_summary_path}')
fold_summary_df


폴드별 성능 요약 저장: .\result\step1_model/fold_performance_summary.csv


,test_target_quarter,test_snapshot_quarter,n_train,n_test,scale_pos_weight,threshold_used,reg_objective,clip_percentile,roc_auc,pr_auc,precision,recall,f1,mae_overall,rmse_overall,mae_zero_baseline,mae_positive_subset,rmse_positive_subset,mae_positive_subset_zero_baseline,n_positive_subset,clip_upper_log,clip_upper_original_scale,clipped_ratio
0,20221,20214,225996,75661,1.5,0.10,huber,0.97,0.853873,0.431787,0.186817,0.898837,0.309339,4.399619,10.298843,1.959041,17.982977,31.079046,21.544041,6880,4.615121,100.0,0.000000
1,20222,20221,301657,75725,1.0,0.10,huber,0.97,0.864834,0.437856,0.180244,0.874432,0.298881,3.721338,9.349476,1.625487,17.211414,30.743218,20.718734,5941,4.615121,100.0,0.000000
2,20223,20222,377382,75910,1.0,0.10,huber,0.97,0.865562,0.513938,0.278746,0.818897,0.415917,4.087744,11.499086,2.585852,20.007557,33.209226,23.038967,8520,4.615121,100.0,0.000132
3,20224,20223,453292,75852,1.5,0.10,huber,0.97,0.864268,0.488221,0.222291,0.865009,0.353690,4.262760,10.341665,2.125389,17.514056,30.455053,21.441016,7519,4.615121,100.0,0.000000
4,20231,20224,529144,76098,1.5,0.10,huber,0.97,0.860512,0.512645,0.242226,0.876538,0.379562,5.025464,11.720148,2.656811,18.454452,32.025231,23.027107,8780,4.615121,100.0,0.000000
5,20232,20231,605242,76156,1.5,0.10,huber,0.97,0.864564,0.513170,0.237644,0.867332,0.373069,4.599887,11.226512,2.453451,18.828918,32.257530,22.741602,8216,4.615121,100.0,0.000000
6,20233,20232,681398,76267,2.0,0.10,huber,0.97,0.870004,0.517895,0.223135,0.907225,0.358175,5.009670,11.107332,2.396187,17.482847,30.738872,22.191864,8235,4.615121,100.0,0.000000
7,20234,20233,757665,76399,1.5,0.10,huber,0.97,0.872563,0.517079,0.209574,0.909766,0.340671,4.518142,10.560726,2.172240,17.186326,30.581559,21.608984,7680,4.615121,100.0,0.000000
8,20241,20234,834064,76504,1.0,0.10,huber,0.97,0.862285,0.519482,0.277288,0.846154,0.417696,4.783279,11.877780,2.807958,19.244354,32.372171,23.208729,9256,4.615121,100.0,0.000000
9,20242,20241,910568,76272,1.0,0.10,huber,0.97,0.864274,0.501570,0.221592,0.904144,0.355947,5.143652,11.166809,2.415080,17.816074,30.454676,22.322225,8252,4.615121,100.0,0.000000


In [58]:
print('=== 폴드 평균 성능 ===')
print(fold_summary_df[['roc_auc', 'pr_auc', 'precision', 'recall', 'f1',
                        'mae_overall', 'mae_positive_subset']].mean(numeric_only=True).round(4))
print()
print('=== mae_overall / mae_zero_baseline 비율 (v5=1.658배였음, 낮을수록 개선) ===')
mae_ratio = (fold_summary_df['mae_overall'] / fold_summary_df['mae_zero_baseline'])
print('폴드 평균 mae_ratio_vs_baseline:', round(mae_ratio.mean(), 4))
print()
print('=== PR-AUC 비교 (v4=0.5331 / v5=0.5226이었음 — v6에서 회복 목표) ===')
print('폴드 평균 PR-AUC:', round(fold_summary_df['pr_auc'].mean(), 4))
print()
print('=== mae_positive_subset 비교 (v4=18.97 / v5=18.98이었음, 낮을수록 개선) ===')
print('폴드 평균 mae_positive_subset:', round(fold_summary_df['mae_positive_subset'].mean(), 4))
print()
print('=== 폴드별로 선택된 scale_pos_weight / threshold ===')
print(fold_summary_df[['test_target_quarter', 'scale_pos_weight', 'threshold_used']].to_string(index=False))
print()
print('=== 폴드별로 선택된 clip_percentile ===')
print(fold_summary_df[['test_target_quarter', 'reg_objective', 'clip_percentile']].to_string(index=False))


=== 폴드 평균 성능 ===
roc_auc                 0.8653
pr_auc                  0.4973
precision               0.2323
recall                  0.8693
f1                      0.3650
mae_overall             4.5931
mae_positive_subset    18.1440
dtype: float64

=== mae_overall / mae_zero_baseline 비율 (v5=1.658배였음, 낮을수록 개선) ===
폴드 평균 mae_ratio_vs_baseline: 2.0045

=== PR-AUC 비교 (v4=0.5331 / v5=0.5226이었음 — v6에서 회복 목표) ===
폴드 평균 PR-AUC: 0.4973

=== mae_positive_subset 비교 (v4=18.97 / v5=18.98이었음, 낮을수록 개선) ===
폴드 평균 mae_positive_subset: 18.144

=== 폴드별로 선택된 scale_pos_weight / threshold ===
 test_target_quarter  scale_pos_weight  threshold_used
               20221               1.5            0.10
               20222               1.0            0.10
               20223               1.0            0.10
               20224               1.5            0.10
               20231               1.5            0.10
               20232               1.5            0.10
               20233               2

In [59]:
print('=== 폴드별 클리핑 상한(원래 스케일 환산) 및 발동 비율 ===')
print(fold_summary_df[['test_target_quarter', 'clip_upper_original_scale', 'clipped_ratio']].to_string(index=False))
print()
print('평균 클리핑 발동 비율(테스트셋 전체 중, v5=0.003%였음):', fold_summary_df['clipped_ratio'].mean().round(6))


=== 폴드별 클리핑 상한(원래 스케일 환산) 및 발동 비율 ===
 test_target_quarter  clip_upper_original_scale  clipped_ratio
               20221                      100.0       0.000000
               20222                      100.0       0.000000
               20223                      100.0       0.000132
               20224                      100.0       0.000000
               20231                      100.0       0.000000
               20232                      100.0       0.000000
               20233                      100.0       0.000000
               20234                      100.0       0.000000
               20241                      100.0       0.000000
               20242                      100.0       0.000000
               20243                      100.0       0.000000
               20244                      100.0       0.000000

평균 클리핑 발동 비율(테스트셋 전체 중, v5=0.003%였음): 1.1e-05


## 6. 1단계 위험점수 테이블 (2단계 입력 재료)

In [60]:
oof_predictions_df = pd.concat(oof_prediction_tables, ignore_index=True)
oof_predictions_path = f'{RESULT_DIR}/stage1_risk_score_table.csv'
oof_predictions_df.to_csv(oof_predictions_path, index=False, encoding='utf-8-sig')

print(f'1단계 위험점수 테이블 저장: {oof_predictions_path}')
print('shape:', oof_predictions_df.shape)
oof_predictions_df.head(10)


1단계 위험점수 테이블 저장: .\result\step1_model/stage1_risk_score_table.csv
shape: (912902, 10)


,상권_코드,서비스_업종_코드,예측_대상_분기,실제_폐업률,예측_폐업위험점수,p_nonzero,p_nonzero_raw,threshold_used,expected_if_positive,expected_if_positive_raw
0,3001491,CS100001,20221,3.0,3.162996,0.987500,0.874695,0.1,3.203034,3.203034
1,3001491,CS100002,20221,5.0,3.271248,0.433333,0.534752,0.1,7.549034,7.549034
2,3001491,CS100003,20221,4.0,2.334103,0.378858,0.492617,0.1,6.160885,6.160885
3,3001491,CS100004,20221,1.0,3.201211,0.987500,0.874695,0.1,3.241732,3.241732
4,3001491,CS100005,20221,0.0,2.883675,0.533480,0.627259,0.1,5.405407,5.405407
5,3001491,CS100006,20221,0.0,2.593367,0.378858,0.465655,0.1,6.845214,6.845214
6,3001491,CS100007,20221,0.0,4.707471,0.259612,0.364340,0.1,18.132747,18.132747
7,3001491,CS100008,20221,5.0,2.757129,0.378858,0.473475,0.1,7.277467,7.277467
8,3001491,CS100009,20221,2.0,3.553734,0.987500,0.874695,0.1,3.598718,3.598718
9,3001491,CS100010,20221,2.0,2.818025,0.930556,0.861017,0.1,3.028326,3.028326


## 6-1. 캘리브레이션 체크

In [61]:
calib_df = oof_predictions_df.copy()
calib_df['실제_발생여부'] = (calib_df['실제_폐업률'] > 0).astype(int)
calib_df['p_nonzero_decile'] = pd.qcut(calib_df['p_nonzero'], 10, labels=False, duplicates='drop')

calib_summary = calib_df.groupby('p_nonzero_decile').agg(
    n=('실제_발생여부', 'size'),
    mean_p_nonzero=('p_nonzero', 'mean'),
    actual_positive_rate=('실제_발생여부', 'mean'),
).reset_index()
calib_summary['abs_gap'] = (calib_summary['mean_p_nonzero'] - calib_summary['actual_positive_rate']).abs()

print('=== p_nonzero 10분위별 캘리브레이션 체크 (isotonic 보정 후) ===')
print(calib_summary.round(4).to_string(index=False))
print()
print('평균 절대 캘리브레이션 오차(낮을수록 좋음):', round(calib_summary['abs_gap'].mean(), 4))


=== p_nonzero 10분위별 캘리브레이션 체크 (isotonic 보정 후) ===
 p_nonzero_decile      n  mean_p_nonzero  actual_positive_rate  abs_gap
                0 100656          0.0010                0.0011   0.0001
                1  84943          0.0021                0.0016   0.0005
                2  89601          0.0050                0.0044   0.0006
                3  91343          0.0216                0.0191   0.0025
                4  90083          0.0479                0.0411   0.0068
                5  91649          0.0818                0.0645   0.0173
                6  91227          0.1291                0.0920   0.0371
                7  90999          0.1954                0.1336   0.0618
                8  91444          0.3005                0.2034   0.0971
                9  90957          0.5477                0.4808   0.0669

평균 절대 캘리브레이션 오차(낮을수록 좋음): 0.0291
